# 6 — Resume Screener
> Hiring teams waste hours reading resumes that don't fit the role. This agent scores every candidate against a job spec and ranks them instantly, so recruiters can focus their time on the people most worth calling.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI


class ResumeScore(BaseModel):
    candidate_name: str
    overall_score: int = Field(ge=1, le=10, description="Overall fit score 1-10")
    tier: Literal["strong_yes", "yes", "maybe", "no"]
    years_experience: int = Field(ge=0, description="Estimated years of relevant experience")
    skills_matched: List[str] = Field(description="Job spec skills found in the resume")
    skills_missing: List[str] = Field(description="Job spec skills not found in the resume")
    standout: str = Field(description="One sentence on the strongest signal in this resume")
    concern: str = Field(description="One sentence on the biggest gap or risk")
    recommended_action: Literal["schedule_interview", "hold_for_review", "pass"]


JOB_SPEC = """
Role: Senior Python Backend Engineer
Company: FinTech startup (Series B, 80 employees)

Required skills:
- Python 5+ years
- REST API design (FastAPI or Django REST)
- PostgreSQL or similar relational database
- Cloud deployment (AWS or GCP)
- Git, CI/CD pipelines

Nice to have:
- Financial domain experience
- Pydantic / data validation
- Docker / Kubernetes
- Team leadership or mentoring

Red flags:
- Less than 3 years total experience
- No production deployment experience
- Only frontend or data science work (we need backend)
"""

SYSTEM_PROMPT = SystemMessage(
    f"""You are a senior technical recruiter screening candidates for a backend engineering role.

Job Specification:
{JOB_SPEC}

For each resume, score the candidate 1-10 against the job spec:
- 9-10: exceptional fit, clear must-interview
- 7-8: strong match, schedule interview
- 5-6: partial match, review more carefully
- 3-4: significant gaps, likely pass
- 1-2: does not meet minimum requirements

Be honest about gaps. Do not inflate scores. Your tier reflects:
  strong_yes  -> score 8-10, all required skills present
  yes         -> score 6-7, most required skills present
  maybe       -> score 4-5, some required skills, notable gaps
  no          -> score 1-3, does not meet requirements"""
)


def create_screener():
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    return SYSTEM_PROMPT | llm.with_structured_output(ResumeScore)


def screen(resume_text: str) -> ResumeScore:
    screener = create_screener()
    return screener.invoke(resume_text)

## The scenario

We're filling a Senior Python Backend Engineer seat at a Series B fintech. Three applications came in over the weekend: a mid-career engineer who pivoted from mobile to backend, a strong DevOps specialist who learned Python on the job, and a former startup CTO who wants to return to hands-on coding. The agent reads each resume against the job spec and returns a ranked shortlist with specific reasons to call — or pass.

In [ ]:
RESUMES = [
    {
        "name": "Maria Santos",
        "text": """Maria Santos -- Backend Engineer
6 years total experience, last 4 years in Python backend.
Started in iOS (Swift), moved to backend after joining a payments startup.
Built REST APIs with FastAPI; PostgreSQL and Redis for data storage.
AWS (Lambda, RDS, S3). Docker and GitHub Actions for CI/CD.
Now owns card processing endpoints handling real-time transaction auth.
Skills: Python, FastAPI, PostgreSQL, AWS, Docker, Git, Redis.""",
    },
    {
        "name": "Derek Okafor",
        "text": """Derek Okafor -- DevOps / Platform Engineer
8 years in infrastructure. Kubernetes, Terraform, GCP, AWS.
Learned Python 3 years ago to write internal automation scripts and glue services.
Some Django work (admin panels and internal tooling only, never public-facing).
No production REST API design or database ownership.
Owns the entire CI/CD pipeline for a 40-person engineering team.
Skills: Python (automation), Django (basic), GCP, AWS, Kubernetes, Terraform, Git.""",
    },
    {
        "name": "Priya Mehta",
        "text": """Priya Mehta -- Engineering Manager / Former CTO
12 years in tech. First 7 years as a Python backend engineer (Django, PostgreSQL, AWS).
Last 5 years in EM and CTO roles -- no hands-on coding since 2019.
Led teams of 10-15 engineers at two Series A fintech startups.
Deep background in system design, architecture, and fintech compliance.
Actively seeking to return to IC engineering.
Skills: Python (pre-2019), Django, PostgreSQL, AWS, Git, system design, team leadership.""",
    },
]

print("RESUME SCREENING RESULTS")
print("=" * 62)

results = []
for r in RESUMES:
    result = screen(r["text"])
    results.append(result)
    print(f"\n{result.candidate_name}")
    print(f"  Score:    {result.overall_score}/10  |  Tier: {result.tier}")
    print(f"  Action:   {result.recommended_action.replace('_', ' ')}")
    print(f"  Matched:  {', '.join(result.skills_matched)}")
    print(f"  Missing:  {', '.join(result.skills_missing) if result.skills_missing else 'None'}")
    print(f"  Standout: {result.standout}")
    print(f"  Concern:  {result.concern}")

print("\n" + "=" * 62)
print("RANKING")
ranked = sorted(results, key=lambda x: x.overall_score, reverse=True)
for i, r in enumerate(ranked, 1):
    print(f"  {i}. {r.candidate_name:22} {r.overall_score}/10  {r.tier:12}  -> {r.recommended_action.replace('_', ' ')}")

## Use your own data

Replace the input above with:
- Your actual job spec (edit the `JOB_SPEC` string in the logic cell above)
- Your real candidate resumes (replace the entries in `RESUMES` with pasted resume text)

The agent returns a ranked shortlist with scores, matched and missing skills, a standout signal, and a specific concern for each candidate.